# Testing 11 · Prostate IHC extraction (rule-based, OncoPanel specimens)

This runs or loads the two steps of `COMPASS/prostate_ihc`:

1. **`extract_prostate_ihc`** builds the prostate cohort (registry C61, malignant prostate pathology, or prostate OncoPanel specimens), the
   long table of panel stain calls (lesional / background, plus supplement calls such as PTEN / RB1 loss), and one row
   per pathology specimen with marker calls, `ihc_phenotype` and `prostate_evidence`. It writes the
   `ihc_extraction/intermediate/*.parquet` files.
2. **`oncopanel_ihc`** matches the OncoPanel prostate specimens (`CDM_SPEC_CON_ids`) to those reports by
   normalised accession. It writes `ihc_extraction/results/oncopanel_prostate_specimen_ihc.csv` and
   `oncopanel_prostate_ihc_long.csv`.

Notebook `text_interpretability/post_landmark/11_LR_for_IHC.ipynb` uses the step-2 CSVs as its outcomes.

With `RERUN_EXTRACTION = False` (the default), the notebook only loads the existing outputs. With `True`, it reruns
both steps from the raw PROFILE_DATA inputs and **overwrites** `intermediate/` and `results/`. Rerun both steps after updating extraction logic; existing files are not migrated. The extraction needs
polars ≥ 1.24.

In [ ]:
import os, sys
from pathlib import Path
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'survival_common').is_dir()), None)
if ROOT is None: raise RuntimeError('Run from the repository or COMPASS/notebooks')
sys.path.insert(0, str(ROOT))
import polars as pl
from IPython.display import display
from COMPASS.prostate_ihc import extract_prostate_ihc as ex
from COMPASS.prostate_ihc import oncopanel_ihc as op
from COMPASS.prostate_ihc import paths as ihc_paths
from COMPASS.multivariable_models import ihc_lr as il

# >>> Edit <<<
RERUN_EXTRACTION = False
P = ihc_paths.default_paths()   # env: COMPASS_IHC_EXTRACTION_ROOT, PROFILE_PATHOLOGY_EXTRACTS, PROFILE_PARQUET_DIR

print(f"polars {pl.__version__}")
for k, v in vars(P).items():
    print(f"{k:<17} {v}")

## Step 1: prostate cohort and pathology IHC calls

In [ ]:
if RERUN_EXTRACTION:
    step1 = ex.run(P.pathology_dir, P.parquet_dir, P.intermediate_dir)
else:
    step1 = ex.load(P.intermediate_dir)
for name, df in step1.items():
    print(f"{name:<7} {df.height:>9,} rows")
for name, table in ex.summary_tables(step1).items():
    print(f"\n=== {name} ===")
    display(table.to_pandas())

## Step 2: OncoPanel prostate specimens

In [ ]:
if RERUN_EXTRACTION:
    spec, long = op.run(P.parquet_dir, P.pathology_dir, P.intermediate_dir, P.results_dir)
else:
    spec, long = op.load(P.results_dir)
print(f"{spec.height:,} OncoPanel prostate specimens; {long.height:,} stain rows on matched reports")
for name, table in op.summary_tables(spec).items():
    print(f"\n=== {name} ===")
    display(table.to_pandas())

## What the LR notebook sees

These are the outcome flags and dates `ihc_lr.load_oncopanel_ihc` derives from the two CSVs, and the per-marker stain frequency on matched specimens.

In [ ]:
spec_pd, long_pd = spec.to_pandas(), long.to_pandas()
lr_spec = il.load_oncopanel_ihc(spec_pd, long_pd)
display(lr_spec[["path_matched", "ihc_any", "ne_pos", "nepc_like"]].sum().rename("specimens").to_frame())
print(f"stained specimens without a pathology date: "
      f"{int((lr_spec['ihc_any'] & lr_spec['specimen_date'].isna()).sum()):,}")
display(il.marker_frequency(spec_pd))